# Experimentation Notebook

Use this notebook to explore the dataset, visualize preprocessing steps,
and inspect model predictions interactively. It is **not** required to
run the project — `scripts/train.py` and `app/app.py` are fully
self-contained — this notebook is just a convenient scratchpad.

Run Jupyter from the project root so the imports below resolve:

```
pip install notebook
jupyter notebook notebooks/experimentation.ipynb
```

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import matplotlib.pyplot as plt

from config import CONFIG
from tensorflow import keras

print("Project root:", CONFIG.PROJECT_ROOT)

## 1. Look at a few raw MNIST digits

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

fig, axes = plt.subplots(1, 8, figsize=(16, 2))
for i, ax in enumerate(axes):
    ax.imshow(x_train[i], cmap="gray")
    ax.set_title(str(y_train[i]))
    ax.axis("off")
plt.show()

## 2. Visualize the custom preprocessing pipeline on a sample image

If you've collected custom data with `scripts/collect_data.py`, point
`sample_path` at one of the saved PNGs to see each pipeline stage.

In [ ]:
import cv2
from src.preprocessing.image_processing import (
    to_grayscale, reduce_noise, binarize, segment_digits
)

sample_path = CONFIG.RAW_DATA_DIR / "7" / "example.png"  # update to a real file

if sample_path.exists():
    frame = cv2.imread(str(sample_path))
    gray = to_grayscale(frame)
    blurred = reduce_noise(gray)
    binary = binarize(blurred)
    digits = segment_digits(frame)

    fig, axes = plt.subplots(1, 3 + len(digits), figsize=(4 * (3 + len(digits)), 4))
    axes[0].imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)); axes[0].set_title("Raw ROI")
    axes[1].imshow(gray, cmap="gray"); axes[1].set_title("Grayscale + blur")
    axes[2].imshow(binary, cmap="gray"); axes[2].set_title("Binarized")
    for i, d in enumerate(digits):
        axes[3 + i].imshow(d.image.squeeze(), cmap="gray")
        axes[3 + i].set_title(f"Digit crop {i}")
    for ax in axes:
        ax.axis("off")
    plt.show()
else:
    print(f"No sample found at {sample_path} - collect some data first.")

## 3. Inspect training history (after running scripts/train.py)

In [ ]:
import json

if CONFIG.TRAINING_HISTORY_PATH.exists():
    with open(CONFIG.TRAINING_HISTORY_PATH) as f:
        history = json.load(f)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(history["accuracy"], label="train")
    ax1.plot(history["val_accuracy"], label="val")
    ax1.set_title("Accuracy"); ax1.legend()

    ax2.plot(history["loss"], label="train")
    ax2.plot(history["val_loss"], label="val")
    ax2.set_title("Loss"); ax2.legend()
    plt.show()
else:
    print("No training history found yet - run scripts/train.py first.")